
# 03 — Time Alignment (OHLCV ↔ Quotes)

**Fecha:** 2026-02-03  
**Slice:** AABA 2019-01  
**Objetivo:** Determinar unidades de timestamp, relación timestamp vs participant_timestamp, y alineación temporal entre Quotes y OHLCV 1m; medir cobertura por sesión.


In [15]:
import json
from pathlib import Path
from datetime import datetime

import polars as pl

from src.core.settings import settings

MANIFEST_PATH = Path("data/manifests/r2_slice_AABA_2019_01.json")
CACHE_DIR = Path(settings.DATA_CACHE_DIR)

print("UTC now:", datetime.utcnow().isoformat(), "Z")
print("Manifest:", MANIFEST_PATH)
print("Cache dir:", CACHE_DIR)

assert MANIFEST_PATH.exists(), f"Missing manifest: {MANIFEST_PATH}"
assert CACHE_DIR.exists(), f"Missing cache dir: {CACHE_DIR}"


UTC now: 2026-02-03T17:24:15.017297 Z
Manifest: data/manifests/r2_slice_AABA_2019_01.json
Cache dir: data/cache


/var/folders/p5/jp6dx55n0tl16f0fwb0jj6t00000gn/T/ipykernel_4351/3949660999.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  print("UTC now:", datetime.utcnow().isoformat(), "Z")


In [16]:
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
objs = manifest["objects"]

ohlcv_keys = [o["key"] for o in objs if o["dataset"] == "ohlcv_intraday_1m"]
quote_keys = [o["key"] for o in objs if o["dataset"] == "quotes_p95"]

print("OHLCV files:", len(ohlcv_keys))
print("Quote files:", len(quote_keys))
print("Example OHLCV:", ohlcv_keys[0] if ohlcv_keys else None)
print("Example Quote:", quote_keys[0] if quote_keys else None)

assert len(ohlcv_keys) == 1, "Expected 1 monthly OHLCV parquet in this slice"
assert len(quote_keys) > 0, "Expected at least 1 quotes parquet in this slice"



OHLCV files: 1
Quote files: 21
Example OHLCV: ohlcv_intraday_1m/2019_2025/AABA/year=2019/month=01/minute.parquet
Example Quote: quotes_p95/AABA/year=2019/month=01/day=02/quotes.parquet


In [17]:
ohlcv_path = CACHE_DIR / ohlcv_keys[0]
q_path = CACHE_DIR / quote_keys[0]

df_ohlcv_head = pl.read_parquet(ohlcv_path, n_rows=5)
df_q_head = pl.read_parquet(q_path, n_rows=5)

print("OHLCV columns:", df_ohlcv_head.columns)
print("OHLCV dtypes:", df_ohlcv_head.dtypes)
print("Quotes columns:", df_q_head.columns)
print("Quotes dtypes:", df_q_head.dtypes)

df_q_head.head(3)


OHLCV columns: ['volume', 'vwap', 'open', 'close', 'high', 'low', 't', 'transactions', 'timestamp', 'ticker', 'date', 'minute']
OHLCV dtypes: [Int64, Float64, Float64, Float64, Float64, Float64, Int64, Int64, Datetime(time_unit='ms', time_zone=None), String, Date, String]
Quotes columns: ['ask_exchange', 'ask_price', 'ask_size', 'bid_exchange', 'bid_price', 'bid_size', 'conditions', 'participant_timestamp', 'sequence_number', 'timestamp', 'tape', 'indicators']
Quotes dtypes: [Int64, Float64, Int64, Int64, Float64, Int64, List(Int64), Int64, Int64, Int64, Int64, List(Int64)]


ask_exchange,ask_price,ask_size,bid_exchange,bid_price,bid_size,conditions,participant_timestamp,sequence_number,timestamp,tape,indicators
i64,f64,i64,i64,f64,i64,list[i64],i64,i64,i64,i64,list[i64]
18,57.17,200,15,56.33,700,[1],1546439400000189852,303792,1546439400060097104,3,null
12,57.17,1000,15,56.33,700,[1],1546439400297882330,305701,1546439400297899484,3,null
12,57.17,1000,2,56.38,300,[1],1546439400319048147,306018,1546439400319064390,3,null


**Helpers: inferir unidad timestamp + conversión a datetime**

In [18]:
def infer_time_unit_from_magnitude(x: int) -> str:
    """
    Infer unit from typical epoch magnitude:
    - ns ~ 1e18
    - us ~ 1e15
    - ms ~ 1e12
    - s  ~ 1e9
    """
    ax = abs(int(x))
    if ax >= 10**17:
        return "ns"
    if ax >= 10**14:
        return "us"
    if ax >= 10**11:
        return "ms"
    return "s"


def to_datetime_series(s: pl.Series, unit: str) -> pl.Series:
    # Polars supports pl.from_epoch for s/ms/us/ns
    return pl.from_epoch(s, time_unit=unit)


def summarize_int_ts(path: Path, col: str) -> dict:
    df = pl.read_parquet(path, columns=[col])
    stats = df.select(
        pl.col(col).min().alias("min"),
        pl.col(col).max().alias("max"),
        pl.col(col).median().alias("median"),
    ).row(0)
    return {"min": int(stats[0]), "max": int(stats[1]), "median": int(stats[2])}


**Inferir unidad de timestamp y participant_timestamp (un día)**

In [19]:
# Pick one representative quotes file (first in slice)
q0 = CACHE_DIR / quote_keys[0]

ts_stats = summarize_int_ts(q0, "timestamp")
pt_stats = summarize_int_ts(q0, "participant_timestamp")

print("timestamp stats:", ts_stats)
print("participant_timestamp stats:", pt_stats)

ts_unit_guess = infer_time_unit_from_magnitude(ts_stats["median"])
pt_unit_guess = infer_time_unit_from_magnitude(pt_stats["median"])

print("timestamp unit guess:", ts_unit_guess)
print("participant_timestamp unit guess:", pt_unit_guess)


timestamp stats: {'min': 1546439400060097104, 'max': 1546462798096767649, 'median': 1546451565478997248}
participant_timestamp stats: {'min': 1546439400000189852, 'max': 1546473600000000000, 'median': 1546451565479019776}
timestamp unit guess: ns
participant_timestamp unit guess: ns


**Validar unidad probando conversiones (ns/us/ms/s) y viendo fechas coherentes**

In [20]:
def try_units_preview(path: Path, col: str, units=("ns", "us", "ms", "s"), n=3):
    df = pl.read_parquet(path, columns=[col]).head(1000)
    s = df[col]
    mn = int(s.min())
    mx = int(s.max())
    print(f"\n[{col}] raw min={mn} max={mx}")
    for u in units:
        try:
            dmin = to_datetime_series(pl.Series([mn]), u)[0]
            dmax = to_datetime_series(pl.Series([mx]), u)[0]
            print(f"  unit={u}  -> min={dmin}  max={dmax}")
        except Exception as e:
            print(f"  unit={u}  -> ERROR: {e}")

try_units_preview(q0, "timestamp")
try_units_preview(q0, "participant_timestamp")



[timestamp] raw min=1546439400060097104 max=1546439540017731916
  unit=ns  -> min=2019-01-02 14:30:00.060097  max=2019-01-02 14:32:20.017731
  unit=us  -> ERROR: year 50974 is out of range



thread '<unnamed>' (107660) panicked at /Users/runner/.cargo/registry/src/index.crates.io-1949cf8c6b5b557f/chrono-0.4.41/src/naive/datetime/mod.rs:1635:38:
`NaiveDateTime + TimeDelta` overflowed


PanicException: `NaiveDateTime + TimeDelta` overflowed

**Fijar unidades definitivas (edita solo estas 2 líneas si hiciera falta)**

In [ ]:
# Set final units based on the preview above.
TS_UNIT = ts_unit_guess
PTS_UNIT = pt_unit_guess

print("Final TS_UNIT:", TS_UNIT)
print("Final PTS_UNIT:", PTS_UNIT)


Final TS_UNIT: ns
Final PTS_UNIT: ns


**Comparar timestamp vs participant_timestamp: deltas (en microsegundos / milisegundos)**

In [ ]:
# Load a subset (to keep it fast) - adjust n_rows if you want full day
df = pl.read_parquet(q0, columns=["timestamp", "participant_timestamp"]).head(2_000_000)

# Convert deltas to seconds as float for interpretability
# If units differ, we bring both to nanoseconds first then delta, else delta in native.
def to_ns(col: str, unit: str) -> pl.Expr:
    # convert epoch to datetime then to int ns
    return pl.from_epoch(pl.col(col), time_unit=unit).dt.epoch(time_unit="ns")

df2 = df.select(
    to_ns("timestamp", TS_UNIT).alias("ts_ns"),
    to_ns("participant_timestamp", PTS_UNIT).alias("pts_ns"),
).with_columns(
    (pl.col("pts_ns") - pl.col("ts_ns")).alias("delta_ns")
)

stats = df2.select(
    pl.col("delta_ns").min().alias("min_ns"),
    pl.col("delta_ns").median().alias("median_ns"),
    pl.col("delta_ns").quantile(0.95).alias("p95_ns"),
    pl.col("delta_ns").quantile(0.99).alias("p99_ns"),
    pl.col("delta_ns").max().alias("max_ns"),
    (pl.col("delta_ns") < 0).mean().alias("neg_ratio"),
).to_dict(as_series=False)

stats


{'min_ns': [-59907252],
 'median_ns': [-194556.0],
 'p95_ns': [-16012.0],
 'p99_ns': [-15278.0],
 'max_ns': [34199126252631],
 'neg_ratio': [0.9999060138598228]}

**Decidir eje temporal para el resto del notebook**

In [ ]:
# Choose the time axis to use for alignment and coverage.
# Default: consolidated "timestamp". If you find participant_timestamp is better, switch.
TIME_AXIS = "timestamp"  # or "participant_timestamp"

print("TIME_AXIS:", TIME_AXIS)


TIME_AXIS: timestamp


**Normalización quotes + bucketing a minuto NY**

In [ ]:
def normalize_quotes(df: pl.DataFrame) -> pl.DataFrame:
    out = df
    if "bid" not in out.columns and "bid_price" in out.columns:
        out = out.rename({"bid_price": "bid"})
    if "ask" not in out.columns and "ask_price" in out.columns:
        out = out.rename({"ask_price": "ask"})
    return out


def quotes_to_minute_buckets(path: Path) -> pl.DataFrame:
    cols = ["bid_price", "ask_price", "timestamp", "participant_timestamp"]
    df = pl.read_parquet(path, columns=[c for c in cols if c in pl.read_parquet(path, n_rows=1).columns])
    df = normalize_quotes(df)

    unit = TS_UNIT if TIME_AXIS == "timestamp" else PTS_UNIT
    ts_col = "timestamp" if TIME_AXIS == "timestamp" else "participant_timestamp"

    # Convert to NY time and floor to minute
    df = df.with_columns(
        pl.from_epoch(pl.col(ts_col), time_unit=unit)
        .dt.replace_time_zone("UTC")
        .dt.convert_time_zone("America/New_York")
        .alias("dt_ny")
    ).with_columns(
        pl.col("dt_ny").dt.truncate("1m").alias("minute_ny")
    )

    # Keep last quote in the minute (by dt_ny)
    df = df.sort("dt_ny").group_by("minute_ny").agg(
        pl.last("bid").alias("bid"),
        pl.last("ask").alias("ask"),
        pl.count().alias("n_quotes_in_minute"),
    )

    return df


**OHLCV: convertir a minuto NY y preparar tabla por minuto**

Nota: aquí asumimos que OHLCV tiene una columna temporal detectable. Si no sabes el nombre, lo inspeccionamos y ajustamos.

In [ ]:
# --- OHLCV: detect timestamp column + convert to NY minute buckets (robust) ---

ohlcv_schema = pl.read_parquet(ohlcv_path, n_rows=1).schema
print("OHLCV schema:", ohlcv_schema)

# Detect candidate timestamp column
for candidate in ["ts", "timestamp", "datetime", "time", "t"]:
    if candidate in ohlcv_schema:
        OHLCV_TS_COL = candidate
        break
else:
    raise ValueError(f"No OHLCV timestamp column found. Columns: {list(ohlcv_schema.keys())}")

print("OHLCV_TS_COL:", OHLCV_TS_COL)
print("OHLCV_TS_DTYPE:", ohlcv_schema[OHLCV_TS_COL])

df_ohlcv = pl.read_parquet(ohlcv_path)

ts_dtype = ohlcv_schema[OHLCV_TS_COL]

if ts_dtype in (pl.Int64, pl.Int32, pl.UInt64, pl.UInt32):
    # epoch integer
    df_ohlcv = df_ohlcv.with_columns(
        pl.from_epoch(pl.col(OHLCV_TS_COL), time_unit=TS_UNIT)
        .dt.replace_time_zone("UTC")
        .dt.convert_time_zone("America/New_York")
        .alias("dt_ny")
    )
elif ts_dtype in (pl.Datetime,):
    # datetime without timezone; assume UTC then convert
    df_ohlcv = df_ohlcv.with_columns(
        pl.col(OHLCV_TS_COL)
        .dt.replace_time_zone("UTC")
        .dt.convert_time_zone("America/New_York")
        .alias("dt_ny")
    )
else:
    raise TypeError(f"Unsupported OHLCV timestamp dtype: {ts_dtype}")

df_ohlcv = df_ohlcv.with_columns(
    pl.col("dt_ny").dt.truncate("1m").alias("minute_ny")
)

df_ohlcv.select(["minute_ny", "open", "high", "low", "close", "volume"]).head()


OHLCV schema: Schema({'volume': Int64, 'vwap': Float64, 'open': Float64, 'close': Float64, 'high': Float64, 'low': Float64, 't': Int64, 'transactions': Int64, 'timestamp': Datetime(time_unit='ms', time_zone=None), 'ticker': String, 'date': Date, 'minute': String})
OHLCV_TS_COL: timestamp
OHLCV_TS_DTYPE: Datetime(time_unit='ms', time_zone=None)


minute_ny,open,high,low,close,volume
"datetime[ms, America/New_York]",f64,f64,f64,f64,i64
2019-01-02 08:26:00 EST,57.0,57.0,57.0,57.0,1000
2019-01-02 08:47:00 EST,57.16,57.16,57.16,57.16,188
2019-01-02 08:50:00 EST,57.94,57.94,57.94,57.94,11114
2019-01-02 09:30:00 EST,56.78,56.97,56.72,56.72,22833
2019-01-02 09:31:00 EST,56.74,56.84,56.57,56.59,21724


**Cobertura por sesión: OHLCV (09:30–16:00 NY)**

In [21]:
SESSION_START = (9, 30)
SESSION_END = (16, 0)

df_ohlcv_session = (
    df_ohlcv.with_columns(
        pl.col("minute_ny").dt.date().alias("date_ny"),
        pl.col("minute_ny").dt.hour().alias("hour"),
        pl.col("minute_ny").dt.minute().alias("minute"),
    )
    # start >= 09:30
    .filter(
        (pl.col("hour") > SESSION_START[0])
        | ((pl.col("hour") == SESSION_START[0]) & (pl.col("minute") >= SESSION_START[1]))
    )
    # end < 16:00
    .filter(
        (pl.col("hour") < SESSION_END[0])
        | ((pl.col("hour") == SESSION_END[0]) & (pl.col("minute") < SESSION_END[1]))
    )
)

ohlcv_cov = (
    df_ohlcv_session.group_by("date_ny")
    .agg(pl.n_unique("minute_ny").alias("minutes_present"))
    .with_columns((pl.col("minutes_present") / 390.0).alias("coverage"))
    .sort("date_ny")
)

print("OHLCV coverage (first 10 days):")
display(ohlcv_cov.head(10))

print("\nOHLCV coverage summary:")
display(ohlcv_cov.select("coverage").describe())



OHLCV coverage (first 10 days):


date_ny,minutes_present,coverage
date,u32,f64
2019-01-02,390,1.0
2019-01-03,390,1.0
2019-01-04,390,1.0
2019-01-07,390,1.0
2019-01-08,390,1.0
2019-01-09,390,1.0
2019-01-10,390,1.0
2019-01-11,389,0.997436
2019-01-14,389,0.997436



OHLCV coverage summary:


statistic,coverage
str,f64
"""count""",21.0
"""null_count""",0.0
"""mean""",0.999512
"""std""",0.001032
"""min""",0.997436
"""25%""",1.0
"""50%""",1.0
"""75%""",1.0
"""max""",1.0


In [23]:
def quotes_coverage_for_key(key: str) -> dict:
    p = CACHE_DIR / key
    qm = quotes_to_minute_buckets(p).with_columns(
        pl.col("minute_ny").dt.date().alias("date_ny"),
        pl.col("minute_ny").dt.hour().alias("hour"),
        pl.col("minute_ny").dt.minute().alias("minute"),
    )

    qm_sess = qm.filter(
        (pl.col("hour") > SESSION_START[0])
        | ((pl.col("hour") == SESSION_START[0]) & (pl.col("minute") >= SESSION_START[1]))
    ).filter(
        (pl.col("hour") < SESSION_END[0])
        | ((pl.col("hour") == SESSION_END[0]) & (pl.col("minute") < SESSION_END[1]))
    )

    mins = qm_sess.select(pl.n_unique("minute_ny").alias("quote_minutes_present")).row(0)[0]
    date = qm_sess.select(pl.col("date_ny").min().alias("date_ny")).row(0)[0]

    return {
        "key": key,
        "date_ny": date,
        "quote_minutes_present": int(mins),
        "coverage": float(mins) / 390.0,
    }

rows = [quotes_coverage_for_key(k) for k in quote_keys]
quotes_cov = pl.DataFrame(rows).sort("date_ny")

print("Quotes coverage (first 10 days):")
display(quotes_cov.head(10))

print("\nQuotes coverage summary:")
display(quotes_cov.select("coverage").describe())



/var/folders/p5/jp6dx55n0tl16f0fwb0jj6t00000gn/T/ipykernel_4351/1124169299.py:32: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("n_quotes_in_minute"),


Quotes coverage (first 10 days):


key,date_ny,quote_minutes_present,coverage
str,date,i64,f64
"""quotes_p95/AABA/year=2019/mont…",2019-01-02,390,1.0
"""quotes_p95/AABA/year=2019/mont…",2019-01-03,124,0.317949
"""quotes_p95/AABA/year=2019/mont…",2019-01-04,82,0.210256
"""quotes_p95/AABA/year=2019/mont…",2019-01-07,390,1.0
"""quotes_p95/AABA/year=2019/mont…",2019-01-08,390,1.0
"""quotes_p95/AABA/year=2019/mont…",2019-01-09,390,1.0
"""quotes_p95/AABA/year=2019/mont…",2019-01-10,100,0.25641
"""quotes_p95/AABA/year=2019/mont…",2019-01-11,390,1.0
"""quotes_p95/AABA/year=2019/mont…",2019-01-14,390,1.0



Quotes coverage summary:


statistic,coverage
str,f64
"""count""",21.0
"""null_count""",0.0
"""mean""",0.82381
"""std""",0.323416
"""min""",0.210256
"""25%""",1.0
"""50%""",1.0
"""75%""",1.0
"""max""",1.0


**Alineación OHLCV ↔ Quotes: mid dentro de [low, high] + resumen**

In [25]:
# OHLCV per-minute range
ohlcv_min = df_ohlcv.select("minute_ny", "high", "low")

# Build Quotes per-minute mids by concatenating day files.
SUBSET_DAYS = 5  # 0 = all days
keys_to_use = quote_keys if SUBSET_DAYS == 0 else quote_keys[:SUBSET_DAYS]

qmins = []
for k in keys_to_use:
    qm = quotes_to_minute_buckets(CACHE_DIR / k).with_columns(
        ((pl.col("bid") + pl.col("ask")) / 2.0).alias("mid")
    ).select("minute_ny", "mid", "n_quotes_in_minute")
    qmins.append(qm)

quotes_min = pl.concat(qmins, how="vertical")

# --- IMPORTANT: align join key dtypes (precision + timezone) ---
# Force both to datetime[ns, America/New_York]
TARGET_DT = pl.Datetime("ns", time_zone="America/New_York")

ohlcv_min = ohlcv_min.with_columns(pl.col("minute_ny").cast(TARGET_DT))
quotes_min = quotes_min.with_columns(pl.col("minute_ny").cast(TARGET_DT))

# Join
joined = ohlcv_min.join(quotes_min, on="minute_ny", how="inner")

EPS = 1e-9
res = joined.with_columns(
    ((pl.col("mid") < (pl.col("low") - EPS)) | (pl.col("mid") > (pl.col("high") + EPS))).alias("violation")
)

summary = res.select(
    pl.len().alias("minutes_joined"),
    pl.sum("violation").alias("violations"),
    (pl.sum("violation") / pl.len()).alias("violation_ratio"),
    pl.mean("n_quotes_in_minute").alias("avg_quotes_per_minute"),
)

print("Alignment summary (mid-in-range):")
display(summary)



/var/folders/p5/jp6dx55n0tl16f0fwb0jj6t00000gn/T/ipykernel_4351/1124169299.py:32: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("n_quotes_in_minute"),


Alignment summary (mid-in-range):


minutes_joined,violations,violation_ratio,avg_quotes_per_minute
u32,u32,f64,f64
1376,266,0.193314,421.324855


**Violations por día + resumen (sin Expr.describe)**

In [26]:
res_by_day = (
    res.with_columns(pl.col("minute_ny").dt.date().alias("date_ny"))
    .group_by("date_ny")
    .agg(
        pl.len().alias("minutes_joined"),
        pl.sum("violation").alias("violations"),
        (pl.sum("violation") / pl.len()).alias("violation_ratio"),
    )
    .sort("date_ny")
)

print("Violations by day (first 10):")
display(res_by_day.head(10))

print("\nViolation ratio summary:")
display(res_by_day.select("violation_ratio").describe())


Violations by day (first 10):


date_ny,minutes_joined,violations,violation_ratio
date,u32,u32,f64
2019-01-02,390,63,0.161538
2019-01-03,124,27,0.217742
2019-01-04,82,12,0.146341
2019-01-07,390,83,0.212821
2019-01-08,390,81,0.207692



Violation ratio summary:


statistic,violation_ratio
str,f64
"""count""",5.0
"""null_count""",0.0
"""mean""",0.189227
"""std""",0.03285
"""min""",0.146341
"""25%""",0.161538
"""50%""",0.207692
"""75%""",0.212821
"""max""",0.217742


**Guardar artefactos (evidencia) + prints**

In [27]:
OUT = Path("runs/data_quality/AABA_2019_01/time_alignment")
OUT.mkdir(parents=True, exist_ok=True)

ohlcv_cov.write_parquet(OUT / "ohlcv_session_coverage.parquet")
quotes_cov.write_parquet(OUT / "quotes_session_coverage.parquet")

# res_by_day existe solo si ejecutaste la celda 15
res_by_day.write_parquet(OUT / "mid_in_range_by_day.parquet")

print("Saved artifacts:")
print("-", OUT / "ohlcv_session_coverage.parquet")
print("-", OUT / "quotes_session_coverage.parquet")
print("-", OUT / "mid_in_range_by_day.parquet")


Saved artifacts:
- runs/data_quality/AABA_2019_01/time_alignment/ohlcv_session_coverage.parquet
- runs/data_quality/AABA_2019_01/time_alignment/quotes_session_coverage.parquet
- runs/data_quality/AABA_2019_01/time_alignment/mid_in_range_by_day.parquet
